In [2]:
from pathlib import Path

MODEL_ID = "google/gemma-4-E2B-it"
WEIGHTS  = Path("model_weights")

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run our PyTorch-side tokenizer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.tokenization import GemmaTokenizer

tok = GemmaTokenizer(WEIGHTS / "tokenizer.json")

s = "Explain MoE in transformers in 3 sentences."
ids = tok.encode(s)

print(f"vocab_size = {tok.vocab_size:,}")
print(f"ids ({len(ids)}): {ids[:10]} ...")

for tid, piece in tok.pretty(ids)[:10]:
    print(f"  {tid:>6}  {piece!r}")

print(f"\nroundtrip: {tok.decode(ids)!r}")

vocab_size = 262,144
ids (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
  155122  'Explain'
    7945  '▁Mo'
  236788  'E'
     528  '▁in'
   39687  '▁transformers'
     528  '▁in'
  236743  '▁'
  236800  '3'
   23974  '▁sentences'
  236761  '.'

roundtrip: 'Explain MoE in transformers in 3 sentences.'


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed via OUR GemmaEmbedding  (pure nn.Embedding lookup)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding import GemmaEmbedding

emb  = GemmaEmbedding.from_safetensors(WEIGHTS / "model.safetensors")
ours = emb(torch.tensor(ids, dtype=torch.long))

print(f"weight : {tuple(emb.embed.weight.shape)}  dtype={emb.embed.weight.dtype}")
print(f"vecs   : {tuple(ours.shape)}")
print(f"ours[0, :6] = {ours[0, :6].tolist()}")

weight : (262144, 1536)  dtype=torch.bfloat16
vecs   : (10, 1536)
ours[0, :6] = [-0.1123046875, 0.1826171875, -0.08251953125, -0.7265625, 2.171875, -1.484375]


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Image + audio soft-token embeddings (towers borrowed from HF for now)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.embedding import GemmaVisionProjector, GemmaAudioProjector

shard = WEIGHTS / "model.safetensors"
vproj = GemmaVisionProjector.from_safetensors(shard)
aproj = GemmaAudioProjector.from_safetensors(shard)

img_embeds = vproj.embed_file("test_data/Moon.jpg",  MODEL_ID)
aud_embeds = aproj.embed_file("test_data/Chats.mp3", MODEL_ID)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Inspect image + audio soft-token embeddings (batch-dim agnostic)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f"Moon.jpg  → {tuple(img_embeds.shape)}  dtype={img_embeds.dtype}")
print(f"  first token : {img_embeds.flatten(0, -2)[0, :6].tolist()}")

print(f"Chats.mp3 → {tuple(aud_embeds.shape)}  dtype={aud_embeds.dtype}")
print(f"  first token : {aud_embeds.flatten(0, -2)[0, :6].tolist()}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/gemma4/feature_extraction_gemma4.py:208: RuntimeWarning: divide by zero encountered in matmul
  mel_spec = np.matmul(magnitude_spec, self.mel_filters)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/g

Moon.jpg  → (260, 1536)  dtype=torch.bfloat16
  first token : [0.9453125, 0.1572265625, -0.88671875, -1.5546875, -0.859375, 0.314453125]
Chats.mp3 → (1, 135, 1536)  dtype=torch.bfloat16
  first token : [-1.90625, 0.037841796875, -3.8125, -1.3203125, -1.7578125, 0.8046875]


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RMSNorm — load the final-norm weight from the shard, run a tensor through.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.RMSnorm import GemmaRMSNorm

norm = GemmaRMSNorm.from_safetensors(
    WEIGHTS / "model.safetensors",
    "model.language_model.norm.weight",
)

torch.manual_seed(0)
x = torch.randn(1, 8, 1536, dtype=torch.bfloat16)
y = norm(x)

print(f"weight  : {tuple(norm.weight.shape)}  dtype={norm.weight.dtype}")
print(f"x       : {tuple(x.shape)}  dtype={x.dtype}")
print(f"y       : {tuple(y.shape)}  dtype={y.dtype}")
print(f"y[0,0,:6] = {y[0, 0, :6].tolist()}")

weight  : (1536,)  dtype=torch.bfloat16
x       : (1, 8, 1536)  dtype=torch.bfloat16
y       : (1, 8, 1536)  dtype=torch.bfloat16
y[0,0,:6] = [20.125, 2.59375, 14.125, 26.125, 7.59375, -20.5]


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RoPE — emit (cos, sin) tables for both layer types.
#  No weights to load; everything is determined by config numbers.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.rope import rope_local, rope_global

seq_len = 8
position_ids = torch.arange(seq_len, dtype=torch.long)[None]   # (1, S)

# global: head_dim=512, theta=1e6, only first 25% of dims rotate
x512 = torch.zeros(1, seq_len, 512, dtype=torch.bfloat16)
cos_g, sin_g = rope_global()(x512, position_ids)

# local: head_dim=256, theta=10_000, all dims rotate
x256 = torch.zeros(1, seq_len, 256, dtype=torch.bfloat16)
cos_l, sin_l = rope_local()(x256, position_ids)

print(f"global cos: {tuple(cos_g.shape)}  cos[0,1,:6] = {cos_g[0, 1, :6].tolist()}")
print(f"global sin: {tuple(sin_g.shape)}  sin[0,1,:6] = {sin_g[0, 1, :6].tolist()}")
print(f"  → tail (no-rope dims) cos[0,1,-4:] = {cos_g[0, 1, -4:].tolist()}  (should be ~1)")
print(f"  → tail (no-rope dims) sin[0,1,-4:] = {sin_g[0, 1, -4:].tolist()}  (should be ~0)")
print()
print(f"local  cos: {tuple(cos_l.shape)}  cos[0,1,:6] = {cos_l[0, 1, :6].tolist()}")
print(f"local  sin: {tuple(sin_l.shape)}  sin[0,1,:6] = {sin_l[0, 1, :6].tolist()}")

global cos: (1, 8, 512)  cos[0,1,:6] = [0.5390625, 0.58203125, 0.625, 0.66015625, 0.69140625, 0.72265625]
global sin: (1, 8, 512)  sin[0,1,:6] = [0.83984375, 0.8125, 0.78125, 0.75, 0.72265625, 0.69140625]
  → tail (no-rope dims) cos[0,1,-4:] = [1.0, 1.0, 1.0, 1.0]  (should be ~1)
  → tail (no-rope dims) sin[0,1,-4:] = [0.0, 0.0, 0.0, 0.0]  (should be ~0)

local  cos: (1, 8, 256)  cos[0,1,:6] = [0.5390625, 0.59765625, 0.6484375, 0.69140625, 0.73046875, 0.765625]
local  sin: (1, 8, 256)  sin[0,1,:6] = [0.83984375, 0.80078125, 0.76171875, 0.72265625, 0.6796875, 0.640625]


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Self-Attention — load layer-0 (local) and layer-4 (global) from the shard
#  and run them on the same hidden tensor as the HF cell.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.attention import GemmaAttention, causal_mask
from architecture.rope      import rope_local, rope_global

torch.manual_seed(0)
B, S = 1, 8
hidden = torch.randn(B, S, 1536, dtype=torch.bfloat16)
position_ids = torch.arange(S, dtype=torch.long)[None]

mask = causal_mask(S, hidden.device, torch.float32)

# ── local layer 0 ────────────────────────────────────────────────
attn_l = GemmaAttention.from_safetensors(
    WEIGHTS / "model.safetensors", layer_idx=0, layer_type="sliding_attention",
)
x256 = torch.zeros(1, S, 256, dtype=torch.bfloat16)
cos_l, sin_l = rope_local()(x256, position_ids)

with torch.no_grad():
    out_l = attn_l(hidden, cos_l, sin_l, attention_mask=mask)

print(f"local  out: {tuple(out_l.shape)}  dtype={out_l.dtype}")
print(f"  out[0, 0, :6] = {out_l[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_l[0,-1, :6].tolist()}")

# ── global layer 4 ───────────────────────────────────────────────
attn_g = GemmaAttention.from_safetensors(
    WEIGHTS / "model.safetensors", layer_idx=4, layer_type="full_attention",
)
x512 = torch.zeros(1, S, 512, dtype=torch.bfloat16)
cos_g, sin_g = rope_global()(x512, position_ids)

with torch.no_grad():
    out_g = attn_g(hidden, cos_g, sin_g, attention_mask=mask)

print(f"\nglobal out: {tuple(out_g.shape)}  dtype={out_g.dtype}")
print(f"  out[0, 0, :6] = {out_g[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_g[0,-1, :6].tolist()}")

local  out: (1, 8, 1536)  dtype=torch.bfloat16
  out[0, 0, :6] = [-0.169921875, 3.4375, -2.96875, -0.03759765625, 1.984375, -2.015625]
  out[0,-1, :6] = [-1.7421875, 1.140625, -0.95703125, 0.765625, 1.0, 1.9765625]

global out: (1, 8, 1536)  dtype=torch.bfloat16
  out[0, 0, :6] = [2.71875, -2.96875, 1.09375, 3.859375, 0.123046875, -0.1708984375]
  out[0,-1, :6] = [1.3046875, 0.40234375, 0.375, 2.3125, -1.515625, 0.1982421875]


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FFN — load layer-0 (standard 6144) and layer-15 (double-wide 12288)
#  from the shard, run through GeGLU.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.ffn import GemmaFFN

torch.manual_seed(0)
x = torch.randn(1, 8, 1536, dtype=torch.bfloat16)

ffn_0 = GemmaFFN.from_safetensors(WEIGHTS / "model.safetensors", layer_idx=0)
with torch.no_grad():
    y0 = ffn_0(x)
print(f"layer  0  inter={ffn_0.intermediate_size}")
print(f"  y[0, 0, :6] = {y0[0, 0, :6].tolist()}")

ffn_15 = GemmaFFN.from_safetensors(WEIGHTS / "model.safetensors", layer_idx=15)
with torch.no_grad():
    y15 = ffn_15(x)
print(f"\nlayer 15  inter={ffn_15.intermediate_size}")
print(f"  y[0, 0, :6] = {y15[0, 0, :6].tolist()}")

layer  0  inter=6144
  y[0, 0, :6] = [2.9375, -2.0, 2.921875, 2.71875, 4.09375, 7.46875]

layer 15  inter=12288
  y[0, 0, :6] = [2.234375, -9.0625, 7.28125, -7.0, -1.15625, -2.953125]


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Decoder layer — load layer-0 (sliding) and layer-4 (full) and verify
#  bit-parity against HF. We feed `per_layer_input` directly (one layer's
#  256-dim slice); the model-level PLE machinery is tested in the next
#  cell. Use the same seed + tensor convention as the HF parity cell.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.transformer_block import GemmaDecoderLayer
from architecture.attention     import causal_mask
from architecture.rope          import rope_local, rope_global

torch.manual_seed(0)
B, S = 1, 8
hidden = torch.randn(B, S, 1536, dtype=torch.bfloat16)
pli    = torch.randn(B, S,  256, dtype=torch.bfloat16)
position_ids = torch.arange(S, dtype=torch.long)[None]
mask = causal_mask(S, hidden.device, torch.float32)

# ── sliding layer 0 ───────────────────────────
ours_0 = GemmaDecoderLayer.from_safetensors(WEIGHTS / "model.safetensors",
                                            layer_idx=0, layer_type="sliding_attention")
cos_l, sin_l = rope_local()(torch.zeros(1, S, 256, dtype=torch.bfloat16), position_ids)
out_0 = ours_0(hidden, pli, cos_l, sin_l, attention_mask=mask)
print(f"layer  0  out: {tuple(out_0.shape)}  dtype={out_0.dtype}")
print(f"  layer_scalar = {ours_0.layer_scalar.item():.18g}")
print(f"  out[0, 0, :6] = {out_0[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_0[0,-1, :6].tolist()}")

# ── full layer 4 ──────────────────────────────
ours_4 = GemmaDecoderLayer.from_safetensors(WEIGHTS / "model.safetensors",
                                            layer_idx=4, layer_type="full_attention")
cos_g, sin_g = rope_global()(torch.zeros(1, S, 512, dtype=torch.bfloat16), position_ids)
out_4 = ours_4(hidden, pli, cos_g, sin_g, attention_mask=mask)
print(f"\nlayer  4  out: {tuple(out_4.shape)}  dtype={out_4.dtype}")
print(f"  layer_scalar = {ours_4.layer_scalar.item():.18g}")
print(f"  out[0, 0, :6] = {out_4[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_4[0,-1, :6].tolist()}")

# Parity vs HF (load with eager attention so kernels match step-for-step).
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16,
                                          attn_implementation="eager")
hf.eval()
with torch.no_grad():
    hf_out_0 = hf.model.language_model.layers[0](
        hidden_states=hidden.clone(), per_layer_input=pli, shared_kv_states={},
        position_embeddings=(cos_l, sin_l), attention_mask=mask, position_ids=position_ids,
    )
    hf_out_4 = hf.model.language_model.layers[4](
        hidden_states=hidden.clone(), per_layer_input=pli, shared_kv_states={},
        position_embeddings=(cos_g, sin_g), attention_mask=mask, position_ids=position_ids,
    )

print(f"\nbit-equal layer 0 ? {torch.equal(out_0, hf_out_0)}")
print(f"bit-equal layer 4 ? {torch.equal(out_4, hf_out_4)}")

layer  0  out: (1, 8, 1536)  dtype=torch.bfloat16
  layer_scalar = 0.017822265625
  out[0, 0, :6] = [0.361328125, -1.46875, -0.1279296875, 0.189453125, 0.01055908203125, -0.078125]
  out[0,-1, :6] = [0.05615234375, 0.5, -0.0556640625, 0.11962890625, 0.17578125, 0.005157470703125]

layer  4  out: (1, 8, 1536)  dtype=torch.bfloat16
  layer_scalar = 0.498046875
  out[0, 0, :6] = [1.8515625, -4.1875, 1.140625, 0.93359375, 0.1611328125, -1.7421875]
  out[0,-1, :6] = [-0.69140625, -1.2421875, -0.294921875, 0.36328125, -0.6171875, 1.1171875]


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


bit-equal layer 0 ? True
bit-equal layer 4 ? True


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Per-Layer Inputs — model-level PLE machinery.
#  Builds the (B, S, num_layers=35, 256) tensor that feeds every decoder
#  layer's PLE block. Verifies bit-parity against HF's same pipeline.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding     import GemmaEmbedding
from architecture.transformer_block import GemmaPerLayerInputs

torch.manual_seed(0)
input_ids = torch.randint(0, 262144, (1, 8))

ours_emb = GemmaEmbedding.from_safetensors(WEIGHTS / "model.safetensors")
ours_pli = GemmaPerLayerInputs.from_safetensors(WEIGHTS / "model.safetensors")

inputs_embeds   = ours_emb(input_ids)
per_layer_inputs = ours_pli(input_ids, inputs_embeds)

print(f"input_ids        : {tuple(input_ids.shape)}")
print(f"inputs_embeds    : {tuple(inputs_embeds.shape)}  dtype={inputs_embeds.dtype}")
print(f"per_layer_inputs : {tuple(per_layer_inputs.shape)}  dtype={per_layer_inputs.dtype}")
print(f"  layer 0 first row [0,0,0,:6] = {per_layer_inputs[0, 0, 0, :6].tolist()}")
print(f"  layer 4 first row [0,0,4,:6] = {per_layer_inputs[0, 0, 4, :6].tolist()}")

# Parity vs HF.
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16,
                                          attn_implementation="eager")
hf.eval()
lm = hf.model.language_model
with torch.no_grad():
    hf_emb = lm.embed_tokens(input_ids)
    hf_pli = lm.get_per_layer_inputs(input_ids, hf_emb)
    hf_pli = lm.project_per_layer_inputs(hf_emb, hf_pli)

print(f"\nbit-equal embeddings     ? {torch.equal(inputs_embeds, hf_emb)}")
print(f"bit-equal per_layer_inputs ? {torch.equal(per_layer_inputs, hf_pli)}")

input_ids        : (1, 8)
inputs_embeds    : (1, 8, 1536)  dtype=torch.bfloat16
per_layer_inputs : (1, 8, 35, 256)  dtype=torch.bfloat16
  layer 0 first row [0,0,0,:6] = [0.375, -0.20703125, -0.125, 0.08154296875, 2.5625, -0.6640625]
  layer 4 first row [0,0,4,:6] = [-0.265625, -0.65625, -0.43359375, -0.1728515625, 1.6328125, 0.9453125]


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


bit-equal embeddings     ? True
bit-equal per_layer_inputs ? True


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Cumulative through layer N — ours vs HF.
#  Pipeline: input_ids → embedding → PLE precompute → loop layers 0..N
#  with the right (cos, sin) per layer type and the i-th PLE slice. Then
#  compare bit-for-bit against HF's hidden_states[N+1].
#
#  Set N to verify any layer's cumulative output. Cap is 14: layers 15+
#  share K/V from earlier layers (a stack-level concern not yet built),
#  so running through them in isolation would diverge from HF.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding         import GemmaEmbedding
from architecture.ple               import GemmaPerLayerInputs
from architecture.transformer_block import GemmaDecoderLayer
from architecture.attention         import causal_mask
from architecture.rope              import rope_local, rope_global

N = 14    # ← change me. Valid range: [0, 14].
assert 0 <= N <= 14, "Layers 15+ need KV-share routing (stack-level)."

torch.manual_seed(0)
B, S = 1, 8
input_ids    = torch.randint(0, 262144, (B, S))           # same seed as HF cell
position_ids = torch.arange(S, dtype=torch.long)[None]
shard        = WEIGHTS / "model.safetensors"

# 1. Embed and build per_layer_inputs (model-level — runs ONCE).
emb = GemmaEmbedding.from_safetensors(shard)
ple = GemmaPerLayerInputs.from_safetensors(shard)
inputs_embeds    = emb(input_ids)                          # (B, S, 1536)
per_layer_inputs = ple(input_ids, inputs_embeds)           # (B, S, 35, 256)

# 2. RoPE tables once per layer type (cached).
cos_l, sin_l = rope_local()(torch.zeros(1, S, 256, dtype=torch.bfloat16), position_ids)
cos_g, sin_g = rope_global()(torch.zeros(1, S, 512, dtype=torch.bfloat16), position_ids)

# 3. Causal mask. For S=8 this matches the sliding-window mask too.
mask = causal_mask(S, inputs_embeds.device, torch.float32)

# 4. Layer-type pattern (4 sliding : 1 full, repeated). Pull from HF config
#    so we never drift if Google retunes the schedule.
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16,
                                          attn_implementation="eager")
hf.eval()
layer_types = hf.config.text_config.layer_types
print(f"layer_types[:{N+1}] = {layer_types[:N+1]}")

# 5. Loop: feed hidden through layer i, slicing the i-th PLE row.
hidden = inputs_embeds
for i in range(N + 1):
    layer = GemmaDecoderLayer.from_safetensors(shard, layer_idx=i, layer_type=layer_types[i])
    cos, sin = (cos_l, sin_l) if layer_types[i] == "sliding_attention" else (cos_g, sin_g)
    pli_i = per_layer_inputs[:, :, i, :]                   # (B, S, 256)
    with torch.no_grad():
        hidden = layer(hidden, pli_i, cos, sin, attention_mask=mask)

# 6. HF reference: hidden_states[N+1] is the residual after layer N.
with torch.no_grad():
    hf_out = hf.model.language_model(input_ids=input_ids, output_hidden_states=True)
hf_hidden = hf_out.hidden_states[N + 1]

print(f"\nafter layer {N}  ours[0, 0, :6] = {hidden  [0, 0, :6].tolist()}")
print(f"after layer {N}  hf  [0, 0, :6] = {hf_hidden[0, 0, :6].tolist()}")
print(f"\nbit-equal through layer {N} ? {torch.equal(hidden, hf_hidden)}")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

layer_types[:15] = ['sliding_attention', 'sliding_attention', 'sliding_attention', 'sliding_attention', 'full_attention', 'sliding_attention', 'sliding_attention', 'sliding_attention', 'sliding_attention', 'full_attention', 'sliding_attention', 'sliding_attention', 'sliding_attention', 'sliding_attention', 'full_attention']

after layer 14  ours[0, 0, :6] = [-0.63671875, -0.8359375, -0.232421875, -0.1455078125, -0.04638671875, -1.3515625]
after layer 14  hf  [0, 0, :6] = [-0.63671875, -0.8359375, -0.232421875, -0.1455078125, -0.04638671875, -1.3515625]

bit-equal through layer 14 ? True


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FULL-STACK cumulative through layer N — with KV-share routing.
#  Layers 15..34 don't compute their own K/V — they reuse K/V cached from
#  the LAST non-shared layer of the same attention type:
#
#      shared sliding (15,16,17,18,20,21,22,23,25,...,33)  ←  layer 13
#      shared full    (19, 24, 29, 34)                     ←  layer 14
#
#  This cell tracks `shared_kv[13]` and `shared_kv[14]` as the loop runs
#  past those layers, then feeds them into shared layers as `cached_kv`.
#  Bit-equal against HF.hidden_states[N+1] for any N in [0, 34].
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding         import GemmaEmbedding
from architecture.ple               import GemmaPerLayerInputs
from architecture.transformer_block import GemmaDecoderLayer
from architecture.attention         import causal_mask
from architecture.rope              import rope_local, rope_global

N = 34   # ← anywhere in [0, 34]
assert 0 <= N <= 34

torch.manual_seed(0)
B, S = 1, 8
input_ids    = torch.randint(0, 262144, (B, S))
position_ids = torch.arange(S, dtype=torch.long)[None]
shard        = WEIGHTS / "model.safetensors"

# 1. Embed and build per_layer_inputs (model-level — runs ONCE).
emb = GemmaEmbedding.from_safetensors(shard)
ple = GemmaPerLayerInputs.from_safetensors(shard)
inputs_embeds    = emb(input_ids)
per_layer_inputs = ple(input_ids, inputs_embeds)

# 2. RoPE tables once per layer type.
cos_l, sin_l = rope_local()(torch.zeros(1, S, 256, dtype=torch.bfloat16), position_ids)
cos_g, sin_g = rope_global()(torch.zeros(1, S, 512, dtype=torch.bfloat16), position_ids)
mask = causal_mask(S, inputs_embeds.device, torch.float32)

# 3. KV-share routing rule, derived from HF config so we never drift.
from transformers import AutoConfig, AutoModelForCausalLM
cfg = AutoConfig.from_pretrained(MODEL_ID).text_config
layer_types = cfg.layer_types
first_kv    = cfg.num_hidden_layers - cfg.num_kv_shared_layers     # 15
prev        = layer_types[:first_kv]

def kv_source(i):
    """For shared layer i, return the source layer index. None if not shared."""
    if i < first_kv:
        return None
    # Last index in `prev` whose layer_type matches layer_types[i].
    return len(prev) - 1 - prev[::-1].index(layer_types[i])

def is_donor(i):
    """True if i is the LAST non-shared layer of its type — must cache K/V."""
    if i >= first_kv:
        return False
    return i == len(prev) - 1 - prev[::-1].index(layer_types[i])

print(f"first_kv_shared_layer = {first_kv}   (donors: 13={prev[13]}, 14={prev[14]})")

# 4. Loop layers 0..N, caching/routing K/V.
hidden    = inputs_embeds
shared_kv = {}                                                     # {donor_idx: (k, v)}
for i in range(N + 1):
    layer = GemmaDecoderLayer.from_safetensors(shard, layer_idx=i, layer_type=layer_types[i])
    cos, sin = (cos_l, sin_l) if layer_types[i] == "sliding_attention" else (cos_g, sin_g)
    pli_i = per_layer_inputs[:, :, i, :]

    src    = kv_source(i)
    cached = shared_kv[src] if src is not None else None

    with torch.no_grad():
        if is_donor(i):
            hidden, k, v = layer(hidden, pli_i, cos, sin, attention_mask=mask,
                                  cached_kv=cached, return_kv=True)
            shared_kv[i] = (k, v)
        else:
            hidden = layer(hidden, pli_i, cos, sin, attention_mask=mask,
                            cached_kv=cached)

# 5. HF reference: hidden_states[N+1] is the residual after layer N.
hf = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16,
                                          attn_implementation="eager")
hf.eval()
with torch.no_grad():
    hf_out = hf.model.language_model(input_ids=input_ids, output_hidden_states=True)
hf_hidden = hf_out.hidden_states[N + 1]

print(f"\nafter layer {N}  ours[0, 0, :6] = {hidden  [0, 0, :6].tolist()}")
print(f"after layer {N}  hf  [0, 0, :6] = {hf_hidden[0, 0, :6].tolist()}")
print(f"\nbit-equal through layer {N} ? {torch.equal(hidden, hf_hidden)}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  End-to-end logits — ours vs HF.
#  Wraps everything we've built into `GemmaTextModel` and checks bit-parity
#  against HF's `model(input_ids).logits` — covering:
#    - embedding  +  PLE precompute
#    - 35 decoder layers with KV-share routing (layers 15..34)
#    - final RMSNorm
#    - tied LM head  (logits = hidden @ embed_tokens.weight.T)
#    - final softcap (30 * tanh(logits / 30))
#
#  If this passes, every stage is calibrated: any later generation diff
#  can be pinned on decoding strategy, not the model.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.text_model import GemmaTextModel

torch.manual_seed(0)
B, S = 1, 8
input_ids = torch.randint(0, 262144, (B, S))
shard     = WEIGHTS / "model.safetensors"

# Build once: loads the shard exactly once and threads the state dict
# through the component loaders (embedding, PLE, 35 decoder layers, norm).
ours = GemmaTextModel.from_safetensors(shard, model_id=MODEL_ID).eval()

with torch.no_grad():
    ours_logits = ours(input_ids)                                     # (B, S, vocab)

print(f"ours logits : {tuple(ours_logits.shape)}  dtype={ours_logits.dtype}")
print(f"  [0, 0, :6] = {ours_logits[0, 0, :6].tolist()}")
print(f"  argmax at last pos = {ours_logits[0, -1].argmax().item()}")

# HF reference (eager so math paths match).
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16,
                                          attn_implementation="eager").eval()
with torch.no_grad():
    hf_logits = hf(input_ids=input_ids).logits

print(f"\nhf   logits : {tuple(hf_logits.shape)}")
print(f"  [0, 0, :6] = {hf_logits[0, 0, :6].tolist()}")
print(f"  argmax at last pos = {hf_logits[0, -1].argmax().item()}")

print(f"\nbit-equal end-to-end ? {torch.equal(ours_logits, hf_logits)}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Greedy generation — our first real outputs from OUR model.
#  Uses HF's processor only to apply the chat template (so the prompt
#  format matches what the model was tuned on), then runs the sampling
#  loop through OUR `GemmaTextModel`:
#
#      for step in range(max_new):
#          logits   = ours(input_ids)               # (B, S, V)
#          next_id  = logits[:, -1].argmax(-1)      # greedy
#          input_ids = cat([input_ids, next_id])
#          if next_id == eos: break
#
#  No KV cache yet — each step re-prefills the growing prompt. That keeps
#  this cell simple and correct; efficiency is a later pass.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from transformers import AutoProcessor
from architecture.tokenization import GemmaTokenizer
from architecture.text_model   import GemmaTextModel

# Build prompt via HF processor (chat template only — the model is ours).
processor = AutoProcessor.from_pretrained(MODEL_ID)
messages  = [{"role": "user",
              "content": [{"type": "text", "text": "Explain MoE in transformers in 3 sentences."}]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
)
input_ids = inputs["input_ids"]
print(f"prompt ids : {tuple(input_ids.shape)}   last 6 = {input_ids[0, -6:].tolist()}")

# Model + tokenizer from OUR code.
shard = WEIGHTS / "model.safetensors"
ours  = GemmaTextModel.from_safetensors(shard, model_id=MODEL_ID).eval()
tok   = GemmaTokenizer(WEIGHTS / "tokenizer.json")

# EOS ids: Gemma uses <end_of_turn> (106) to close assistant turns, <eos> (1)
# as the SentencePiece sentinel. Stop on either.
eos_ids = {1, 106}
max_new = 64

with torch.no_grad():
    for step in range(max_new):
        logits  = ours(input_ids)                         # (1, S, V)
        next_id = logits[:, -1].argmax(-1, keepdim=True)  # (1, 1)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        if next_id.item() in eos_ids:
            break

generated = input_ids[0, -(step + 1):].tolist()
print(f"\ngenerated {len(generated)} tokens:")
print(f"  ids    : {generated}")
print(f"  decode : {tok.decode(generated)!r}")
